# 🧠 D-SIT: Dopaminergic Spiking Image Transformer
**O(1) Memory, Fully Unsupervised Dopamine Leak, & DA-PSG on CIFAR-100**

---

> **Author:** Siddhesh Uttarwar — Brain-inspired Computing Lab, UCSB

This notebook trains the D-SIT architecture on CIFAR-100 using a Google Colab T4 GPU. With our new E-prop implementation, the network learns dynamic temporal leak factors using (1)$ memory complexity in time!

### Before You Start
1. Go to **Runtime → Change runtime type → T4 GPU**
2. Run all cells sequentially

## Step 1: Verify Device (GPU or TPU)

In [22]:
import torch

print(f"PyTorch version: {torch.__version__}")

# Detect TPU first, then GPU, then CPU
try:
    import torch_xla
    import torch_xla.core.xla_model as xm
    device = xm.xla_device()
    print(f"✅ TPU detected: {device}")
    print(f"torch_xla version: {torch_xla.__version__}")
    DEVICE_TYPE = 'tpu'
except ImportError:
    if torch.cuda.is_available():
        device = torch.device('cuda')
        print(f"✅ GPU detected: {torch.cuda.get_device_name(0)}")
        print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
        DEVICE_TYPE = 'cuda'
    else:
        device = torch.device('cpu')
        print("⚠️  No GPU or TPU detected — running on CPU (slow)")
        DEVICE_TYPE = 'cpu'

print(f"\nActive device: {DEVICE_TYPE.upper()} ({device})")

PyTorch version: 2.9.0+cpu
✅ TPU detected: xla:0
torch_xla version: 2.9.0

Active device: TPU (xla:0)


/tmp/ipykernel_1838/3862776319.py:9: DeprecationWarning: Use torch_xla.device instead
  device = xm.xla_device()


## Step 2: Clone the Repository

In [23]:
!git clone https://github.com/SiddheshUttarwar/PSGD-D-SpikingImageTransformer.git
%cd PSGD-D-SpikingImageTransformer

Cloning into 'PSGD-D-SpikingImageTransformer'...
remote: Enumerating objects: 91, done.
remote: Counting objects: 100% (91/91), done.
remote: Compressing objects: 100% (68/68), done.
remote: Total 91 (delta 58), reused 55 (delta 23), pack-reused 0 (from 0)
Receiving objects: 100% (91/91), 45.52 KiB | 2.84 MiB/s, done.
Resolving deltas: 100% (58/58), done.
/content/PSGD-D-SpikingImageTransformer/PSGD-D-SpikingImageTransformer/PSGD-D-SpikingImageTransformer/PSGD-D-SpikingImageTransformer


## Step 3: Install Dependencies

In [24]:
!pip install datasets tqdm -q
# torch_xla is pre-installed on Colab TPU runtimes (Runtime → Change runtime type → TPU).
# If you get an ImportError for torch_xla on a non-Colab TPU environment, run:
# !pip install torch_xla -q

## Step 4: Smoke Test — Verify Model Builds

In [25]:
import sys
sys.path.insert(0, 'd-sit')

from model import DSIT
from optimizer import ProximalAdamW
import torch
import torch.nn as nn

# Build the model with CIFAR-100 config
model = DSIT(
    num_classes=100,
    embed_dim=256,
    depth=8,
    num_heads=8,
    T=4,
    img_size=32
)

# Quick forward pass test
x = torch.randn(2, 3, 32, 32)
logits = model(x)
print(f"✅ Forward pass OK — Output shape: {logits.shape}")

# Quick backward pass test
labels = torch.tensor([5, 10])
loss = nn.CrossEntropyLoss()(logits, labels)
loss.backward()
print(f"✅ Backward pass OK — Loss: {loss.item():.4f}")

# Verify dopamine tracker — update() takes logits only (no labels)
model.d_tracker.update(logits.detach())
print(f"✅ Dopamine D(t): {model.d_tracker.get_D():.6f} (near 0 = exploration mode)")

param_count = sum(p.numel() for p in model.parameters()) / 1e6
print(f"✅ Model size: {param_count:.2f}M parameters")

del model, x, logits  # Free memory
if torch.cuda.is_available():
    torch.cuda.empty_cache()

✅ Forward pass OK — Output shape: torch.Size([2, 100])
✅ Backward pass OK — Loss: 3.8514
✅ Dopamine D(t): 0.000000 (near 0 = exploration mode)
✅ Model size: 7.78M parameters


## Step 5: Train on CIFAR-100

This will:
- Download CIFAR-100 from HuggingFace (~170MB)
- Build a D-SIT with `embed_dim=256, depth=8, heads=8` (~7.7M params)
- Train with mixed precision + gradient accumulation (effective batch=64)
- Save `dsit_best.pth` and `dsit_latest.pth` for checkpointing
- Log head pruning status every 5 epochs

In [26]:
!python d-sit/train.py \
    --dataset cifar100 \
    --img_size 32 \
    --batch_size 128 \
    --accum_steps 8 \
    --embed_dim 384 \
    --depth 8 \
    --num_heads 12 \
    --T 4 \
    --lr 1e-3 \
    --prox_lambda 1e-4 \
    --epochs 100

/content/PSGD-D-SpikingImageTransformer/PSGD-D-SpikingImageTransformer/PSGD-D-SpikingImageTransformer/PSGD-D-SpikingImageTransformer/d-sit/tokenizer.py:8: SyntaxWarning: invalid escape sequence '\D'
  Assigns a continuous learnable delay \Delta t_d to each of the D=768 dimensions.
[warn] TPU unavailable (TPU initialization failed: open(/dev/vfio/0): Device or resource busy: Device or resource busy; Couldn't open iommu group /dev/vfio/0), falling back to GPU/CPU.
Device: CPU (cpu)
Loading cifar100 (img_size=32)...
`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'cifar100' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'cifar100' isn't based on a loading script and remove `trust_remote_co

## Step 6 (If Disconnected): Resume Training

If your Colab session disconnects, re-run Steps 1-3, then run this cell:

In [27]:
# Only run this if you got disconnected and need to resume
!python d-sit/train.py \
    --dataset cifar100 \
    --img_size 32 \
    --batch_size 16 \
    --accum_steps 4 \
    --embed_dim 256 \
    --depth 8 \
    --num_heads 8 \
    --T 4 \
    --lr 1e-3 \
    --prox_lambda 1e-4 \
    --epochs 100 \
    --resume dsit_latest.pth

[warn] TPU unavailable (TPU initialization failed: open(/dev/vfio/0): Device or resource busy: Device or resource busy; Couldn't open iommu group /dev/vfio/0), falling back to GPU/CPU.
Device: CPU (cpu)
Loading cifar100 (img_size=32)...
`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'cifar100' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'cifar100' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'cifar100' isn't based on a loading script and rem

## Step 7: Analyze Head Pruning

After training, inspect which attention heads were pruned by the proximal optimizer:

In [28]:
import sys
sys.path.insert(0, 'd-sit')
import torch
from model import DSIT
from optimizer import ProximalAdamW

# Detect device (TPU > GPU > CPU) — mirrors train.py's _get_device()
try:
    import torch_xla.core.xla_model as xm
    device = xm.xla_device()
    print(f"Loading on TPU: {device}")
except ImportError:
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Loading on: {device}")

model = DSIT(num_classes=100, embed_dim=256, depth=8, num_heads=8, T=4, img_size=32).to(device)

ckpt = torch.load('dsit_best.pth', map_location=device)
model.load_state_dict(ckpt['model_state_dict'])
print(f"Loaded best checkpoint from epoch {ckpt['epoch']+1}")
print(f"Best Val Acc: {ckpt['best_val_acc']*100:.2f}%")
print(f"Final D(t): {ckpt['D_t']:.6f}")

# Analyze head norms
print("\n" + "="*50)
print("ATTENTION HEAD PRUNING ANALYSIS")
print("="*50)

total_alive = 0
total_heads = 0
for i, blk in enumerate(model.blocks):
    w = blk.attn.out_proj.weight
    w_reshaped = w.view(8, -1)
    norms = torch.norm(w_reshaped, p=2, dim=1)
    alive = (norms > 1e-6).sum().item()
    total_alive += alive
    total_heads += 8
    status = ' '.join(['🟢' if n > 1e-6 else '🔴' for n in norms])
    print(f"Block {i:2d}: {status}  ({alive}/8 alive, max_norm={norms.max():.4f})")

print(f"\nTotal: {total_alive}/{total_heads} heads alive ({100*total_alive/total_heads:.1f}%)")

Loading on TPU: xla:0


/tmp/ipykernel_1838/2689945738.py:10: DeprecationWarning: Use torch_xla.device instead
  device = xm.xla_device()


FileNotFoundError: [Errno 2] No such file or directory: 'dsit_best.pth'

## Step 8: Save Checkpoint to Google Drive (Optional)

Persist your trained model across Colab sessions:

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
import os

save_dir = '/content/drive/MyDrive/DSIT_checkpoints'
os.makedirs(save_dir, exist_ok=True)

for f in ['dsit_best.pth', 'dsit_latest.pth']:
    if os.path.exists(f):
        shutil.copy(f, os.path.join(save_dir, f))
        print(f"✅ Saved {f} to Google Drive")
    else:
        print(f"⚠️ {f} not found")

print(f"\nCheckpoints saved to: {save_dir}")